# BBC News — Topic-Feature Classification Experiment

Evaluates how well each topic model's document representations support downstream classification.
Each model is fit on the **training set only** and uses `transform()` for test docs — no label leakage.

**Dataset:** SetFit/bbc-news — 2,225 docs, 5 categories (business, entertainment, politics, sport, tech)  
**Split:** stratified 80/20 (1,780 train / 445 test, seed=42)  
**Features:** (n_docs, n_topics) topic-probability matrix from each model  
**Classifiers:** Logistic Regression · SVM-RBF · kNN-cosine  
**Metrics:** Accuracy · Macro F1 · per-class F1

**Kernel:** Python (katm-venv)

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import nltk
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

from katm import KATM
from katm.octis_data_prep import load_dataset as load_octis_dataset

for _r in ["corpora/stopwords", "tokenizers/punkt", "tokenizers/punkt_tab"]:
    try: nltk.data.find(_r)
    except LookupError: nltk.download(_r.split("/")[1], quiet=True)

## Parameters

In [ ]:
N_TOPICS  = 5
SEED      = 42
TEST_SIZE = 0.2
KATM_CONTENT_METHOD = "spacy"

## Stopwords

In [ ]:
_EXTRA_STOP = {
    "edu", "com", "org", "net", "gov", "www", "http", "re", "cc", "writes",
    "article", "subject", "lines", "like", "just", "don", "use", "know",
    "think", "time", "want", "good", "make", "look", "said", "say", "way",
    "does", "did", "got", "ll", "ve", "haven", "isn", "aren", "wasn",
    "doesn", "didn", "wouldn", "couldn", "shouldn",
}
_ALL_STOP  = set(nltk.corpus.stopwords.words("english")) | _EXTRA_STOP
_STOP_LIST = sorted(_ALL_STOP)
print(f"Stopword list size: {len(_ALL_STOP)}")

## Data — stratified 80/20 split

In [ ]:
ds = concatenate_datasets([
    load_dataset("SetFit/bbc-news", split="train"),
    load_dataset("SetFit/bbc-news", split="test"),
])
docs       = [r["text"] for r in ds]
labels_txt = [r["label_text"] for r in ds]
names      = sorted(set(labels_txt))
labels     = np.array([names.index(t) for t in labels_txt])

X_tr, X_te, y_tr, y_te = train_test_split(
    docs, labels, test_size=TEST_SIZE, random_state=SEED, stratify=labels
)
print(f"Total: {len(docs)}  |  Train: {len(X_tr)}  Test: {len(X_te)}")
print(f"Classes: {names}")
for i, n in enumerate(names):
    print(f"  {n}: {(y_tr==i).sum()} train / {(y_te==i).sum()} test")

## Evaluation helpers

In [ ]:
all_results = {}

def eval_classifiers(Xtr, Xte, label):
    clfs = {
        "LogReg":  LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
        "SVM-RBF": SVC(kernel="rbf", C=1.0, random_state=SEED),
        "kNN-cos": KNeighborsClassifier(n_neighbors=5, metric="cosine"),
    }
    print(f"\n{'─'*65}\n  {label}\n{'─'*65}")
    hdr = f"  {'Classifier':<12}  {'Acc':>6}  {'MacroF1':>8}  " + \
          "  ".join(f"{n[:5]:>6}" for n in names)
    print(hdr)
    print(f"  {'─'*12}  {'─'*6}  {'─'*8}  " + "  ".join(["─"*6]*len(names)))
    res = {}
    for clf_name, clf in clfs.items():
        clf.fit(Xtr, y_tr)
        preds = clf.predict(Xte)
        acc = accuracy_score(y_te, preds)
        mf1 = f1_score(y_te, preds, average="macro")
        pf1 = f1_score(y_te, preds, average=None, labels=list(range(len(names))))
        per = "  ".join(f"{v:>6.3f}" for v in pf1)
        print(f"  {clf_name:<12}  {acc:>6.3f}  {mf1:>8.3f}  {per}")
        res[clf_name] = dict(acc=acc, macro_f1=mf1, per_class=pf1, preds=preds)
    all_results[label] = res
    return res

def plot_confusion(preds, title):
    cm = confusion_matrix(y_te, preds, labels=list(range(len(names))))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=[n[:4] for n in names])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Baseline 1 — TF-IDF

In [ ]:
t0 = time.time()
tf = TfidfVectorizer(min_df=3, max_df=0.95, stop_words=_STOP_LIST,
                     token_pattern=r"(?u)\b[a-zA-Z]{4,}\b", sublinear_tf=True)
Xtr_tf = tf.fit_transform(X_tr)
Xte_tf = tf.transform(X_te)
print(f"vocab={Xtr_tf.shape[1]}  {time.time()-t0:.1f}s")
res_tf = eval_classifiers(Xtr_tf, Xte_tf, "TF-IDF")
plot_confusion(res_tf["LogReg"]["preds"], "TF-IDF + LogReg")

## Baseline 2 — Sentence-Transformer embeddings *(upper bound)*

In [ ]:
t0  = time.time()
st  = SentenceTransformer("all-MiniLM-L6-v2")
Xtr_st = st.encode(X_tr, batch_size=64, show_progress_bar=False)
Xte_st = st.encode(X_te, batch_size=64, show_progress_bar=False)
print(f"dim={Xtr_st.shape[1]}  {time.time()-t0:.1f}s")
res_st = eval_classifiers(Xtr_st, Xte_st, "ST-embed (upper bound)")
plot_confusion(res_st["SVM-RBF"]["preds"], "ST-embed + SVM-RBF")

## LDA

In [ ]:
t0  = time.time()
cv  = CountVectorizer(min_df=3, max_df=0.95, stop_words=_STOP_LIST,
                      token_pattern=r"(?u)\b[a-zA-Z]{4,}\b")
Xtr_cv = cv.fit_transform(X_tr)
Xte_cv = cv.transform(X_te)
lda = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=200,
                                 random_state=SEED, n_jobs=-1)
Xtr_lda = lda.fit_transform(Xtr_cv)
Xte_lda = lda.transform(Xte_cv)
print(f"{time.time()-t0:.1f}s  |  shape: {Xtr_lda.shape}")
res_lda = eval_classifiers(Xtr_lda, Xte_lda, f"LDA ({N_TOPICS} topics)")
plot_confusion(res_lda["SVM-RBF"]["preds"], f"LDA ({N_TOPICS} topics) + SVM-RBF")

## NMF (Non-negative Matrix Factorisation)

In [ ]:
t0 = time.time()
_vec_nmf   = TfidfVectorizer(min_df=3, max_df=0.95, stop_words=_STOP_LIST,
                    token_pattern=r"(?u)\b[a-zA-Z]{4,}\b")
Xtr_nmf_tf = _vec_nmf.fit_transform(X_tr)
Xte_nmf_tf = _vec_nmf.transform(X_te)
_vocab_nmf = _vec_nmf.get_feature_names_out()
_nmf       = NMF(n_components=N_TOPICS, random_state=SEED, max_iter=400, init="nndsvda")
Xtr_nmf    = _nmf.fit_transform(Xtr_nmf_tf)
Xte_nmf    = _nmf.transform(Xte_nmf_tf)
print(f"{time.time()-t0:.1f}s  |  shape: {Xtr_nmf.shape}")
print("\nNMF topic words:")
for _i, _comp in enumerate(_nmf.components_):
    _words = [_vocab_nmf[j] for j in _comp.argsort()[:-11:-1]]
    print(f"  Topic {_i}: {', '.join(_words)}")
res_nmf = eval_classifiers(Xtr_nmf, Xte_nmf, f"NMF ({N_TOPICS} topics)")
plot_confusion(res_nmf["LogReg"]["preds"], f"NMF ({N_TOPICS} topics) + LogReg")

## BERTopic

In [ ]:
t0     = time.time()
bt_vec = CountVectorizer(stop_words=_STOP_LIST, token_pattern=r"(?u)\b[a-zA-Z]{4,}\b")
bt     = BERTopic(nr_topics=N_TOPICS, min_topic_size=15,
                  vectorizer_model=bt_vec, verbose=False)
bt.fit(X_tr)
print(f"fit: {time.time()-t0:.1f}s — running approximate_distribution ...")
t1 = time.time()
Xtr_bt, _ = bt.approximate_distribution(X_tr, batch_size=256)
Xte_bt, _ = bt.approximate_distribution(X_te, batch_size=256)
Xtr_bt = np.array(Xtr_bt)
Xte_bt = np.array(Xte_bt)
print(f"approx_dist: {time.time()-t1:.1f}s  |  shape: {Xtr_bt.shape}")
res_bt = eval_classifiers(Xtr_bt, Xte_bt, f"BERTopic ({Xtr_bt.shape[1]} topics)")
plot_confusion(res_bt["LogReg"]["preds"], f"BERTopic ({Xtr_bt.shape[1]} topics) + LogReg")

## KATM

Uses `fit_transform()` on train and `transform()` on test — no label leakage.

In [ ]:
t0   = time.time()
katm = KATM(
    kp_algorithm="keybert", n_keyphrases=10, keybert_ngram_range=(1, 2),
    n_topics=N_TOPICS, min_anchor_df=3, max_anchor_df_ratio=0.4,
    anchor_dedup_threshold=0.85, soft_threshold=0.15, min_df=3,
    top_n_words=10, mmr_diversity=0.2, mmr_max_sim=0.85,
    content_words_method=KATM_CONTENT_METHOD,
)
Xtr_katm = katm.fit_transform(X_tr)
Xte_katm = katm.transform(X_te)
print(f"{time.time()-t0:.1f}s  |  shape: {Xtr_katm.shape}")
print("\nTopic words learned on training set:")
katm.print_topics(n_words=8)
res_katm = eval_classifiers(Xtr_katm, Xte_katm, f"KATM ({N_TOPICS} topics)")
plot_confusion(res_katm["SVM-RBF"]["preds"], f"KATM ({N_TOPICS} topics) + SVM-RBF")

## NeuralLDA & ProdLDA *(OCTIS / AVITM)*

**Note:** OCTIS uses spaCy lemmatisation (min_df=5) and an 85/7.5/7.5 train/val/test split —
different from the 80/20 split used by the methods above.

In [ ]:
_bbc_octis_ds = load_octis_dataset("BBC_News")
_bbc_oc_tr, _bbc_oc_val, _bbc_oc_te = _bbc_octis_ds.get_partitioned_corpus()
_bbc_oc_labels_all  = _bbc_octis_ds.get_labels()
_bbc_oc_label_names = sorted(set(_bbc_oc_labels_all))
_bbc_oc_label2id    = {l: i for i, l in enumerate(_bbc_oc_label_names)}
_n_oc_tr  = len(_bbc_oc_tr)
_n_oc_val = len(_bbc_oc_val)
_bbc_y_tr = np.array([_bbc_oc_label2id[l] for l in _bbc_oc_labels_all[:_n_oc_tr]])
_bbc_y_te = np.array([_bbc_oc_label2id[l] for l in _bbc_oc_labels_all[_n_oc_tr + _n_oc_val:]])
print(f"OCTIS BBC_News: {_n_oc_tr} train / {_n_oc_val} val / {len(_bbc_oc_te)} test")
print(f"Classes: {_bbc_oc_label_names}")
print("Note: OCTIS uses 85/7.5/7.5 split vs 80/20 above — results are indicative, not directly comparable.")

### NeuralLDA

In [ ]:
from octis.models.NeuralLDA import NeuralLDA as _NeuralLDA
t0 = time.time()
try:
    _nlda = _NeuralLDA(num_topics=N_TOPICS, num_epochs=100, batch_size=64, lr=2e-3)
    _nlda_out = _nlda.train_model(_bbc_octis_ds, top_words=10)
    print(f"NeuralLDA: {time.time()-t0:.1f}s")
    print("\nNeuralLDA topic words:")
    for _i, _words in enumerate(_nlda_out["topics"]):
        print(f"  Topic {_i}: {', '.join(_words)}")
    _Xtr_nlda = _nlda_out["topic-document-matrix"].T
    _Xte_nlda = _nlda_out["test-topic-document-matrix"].T
    _y_tr_bk, _y_te_bk, _names_bk = y_tr, y_te, names
    y_tr, y_te, names = _bbc_y_tr, _bbc_y_te, _bbc_oc_label_names
    eval_classifiers(_Xtr_nlda, _Xte_nlda, f"NeuralLDA ({N_TOPICS} topics, OCTIS)")
    y_tr, y_te, names = _y_tr_bk, _y_te_bk, _names_bk
except Exception as _e:
    print(f"NeuralLDA FAILED: {_e}")

### ProdLDA

In [ ]:
from octis.models.ProdLDA import ProdLDA as _ProdLDA
t0 = time.time()
try:
    _plda = _ProdLDA(num_topics=N_TOPICS, num_epochs=100, batch_size=64, lr=2e-3)
    _plda_out = _plda.train_model(_bbc_octis_ds, top_words=10)
    print(f"ProdLDA: {time.time()-t0:.1f}s")
    print("\nProdLDA topic words:")
    for _i, _words in enumerate(_plda_out["topics"]):
        print(f"  Topic {_i}: {', '.join(_words)}")
    _Xtr_plda = _plda_out["topic-document-matrix"].T
    _Xte_plda = _plda_out["test-topic-document-matrix"].T
    _y_tr_bk, _y_te_bk, _names_bk = y_tr, y_te, names
    y_tr, y_te, names = _bbc_y_tr, _bbc_y_te, _bbc_oc_label_names
    eval_classifiers(_Xtr_plda, _Xte_plda, f"ProdLDA ({N_TOPICS} topics, OCTIS)")
    y_tr, y_te, names = _y_tr_bk, _y_te_bk, _names_bk
except Exception as _e:
    print(f"ProdLDA FAILED: {_e}")

## Summary

In [ ]:
print(f"{'Method':<28}  {'LogReg':>8}  {'SVM-RBF':>8}  {'kNN-cos':>8}")
print(f"  (Accuracy / Macro F1)")
print("─" * 65)
for method, res in all_results.items():
    lr = res["LogReg"]; sv = res["SVM-RBF"]; kn = res["kNN-cos"]
    print(f"{method:<28}  {lr['acc']:.3f}/{lr['macro_f1']:.3f}  "
          f"{sv['acc']:.3f}/{sv['macro_f1']:.3f}  "
          f"{kn['acc']:.3f}/{kn['macro_f1']:.3f}")

# Per-class F1 bar chart (LogReg, all methods)
_plot_names = sorted(set(all_results.keys()) - {k for k in all_results if "OCTIS" in k})
x      = np.arange(5)  # 5 BBC categories
width  = 0.15
fig, ax = plt.subplots(figsize=(12, 4))
cmap    = plt.get_cmap("tab10")
for i, method in enumerate(_plot_names):
    pf1 = all_results[method]["LogReg"]["per_class"]
    ax.bar(x + i * width, pf1, width, label=method, color=cmap(i / len(_plot_names)))
ax.set_xticks(x + width * (len(_plot_names) - 1) / 2)
ax.set_xticklabels(sorted(set(labels_txt)))
ax.set_ylabel("F1 score")
ax.set_ylim(0, 1.05)
ax.set_title("Per-class F1 — LogReg classifier")
ax.legend(loc="lower right", fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()